# A Machine Learning Approach to Predict Fraud and Cyber Attacks
### XYZ Cybersecurity — Network Flow Threat Detection

---

**Business problem.** XYZ Cybersecurity is a startup that detects and stops online threats. This
notebook builds and evaluates a machine learning system that flags **fraudulent / malicious network
flows** (intrusions, DoS, brute-force, web attacks, botnet/infiltration traffic) from benign traffic,
using bidirectional flow statistics extracted from packet captures (CICIDS-style features, 78
predictors + 1 label).

**Why this matters.** A missed attack (false negative) can mean a breach, data loss and regulatory
fines. A false alarm (false positive) wastes scarce analyst time and can block legitimate customers.
The model must therefore be optimised for **high recall at an operationally acceptable false-positive
rate**, not for raw accuracy — the data is heavily imbalanced, so accuracy is a misleading metric.

---

## Notebook structure (mapped to the marking scheme)

| # | Section | Marks |
|---|---------|-------|
| 1 | Data Understanding & Preparation, including Feature Engineering | 3 |
| 2 | Model Selection | 3 |
| 3 | Performance Measurement | 3 |
| 4 | Hyperparameter Tuning | 5 |
| 5 | Extra Features & Considerations | 3 |
| 6 | AI Ethics Considerations | 5 |
| 7 | Documentation & Code Quality | 3 |
| | **Total** | **25** |

**How to run:** `Runtime → Run all`. Section 0 installs/verifies dependencies. The loader in
Section 1.1 reads `Dataset1.csv`, `Dataset2.csv`, `Dataset3.csv` from your Google Drive; if Drive is
not available it falls back to local files, and finally to a clearly-labelled **simulated** dataset so
the notebook always executes end-to-end.

---
# 0. Environment, Configuration & Reproducibility

All tunable knobs live in a single `CFG` object so the experiment can be re-run deterministically and
reviewers can see every choice in one place. Random seeds are fixed everywhere.

In [ ]:
# ---------------------------------------------------------------------
# 0.1  Optional dependencies (already present in Colab, installed if missing)
# ---------------------------------------------------------------------
import importlib
import subprocess
import sys


def ensure(package: str, pip_name: str | None = None) -> bool:
    """Import `package`, pip-installing `pip_name` first if it is missing.

    Returns True if the package is importable after this call.
    """
    try:
        importlib.import_module(package)
        return True
    except ImportError:
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", pip_name or package],
                check=True,
            )
            importlib.import_module(package)
            print(f"installed: {pip_name or package}")
            return True
        except Exception as exc:                      # offline runtime, restricted env, ...
            print(f"could not install {pip_name or package} -> {type(exc).__name__}; "
                  "the notebook will skip the optional steps that need it")
            return False


HAS_XGB = ensure("xgboost")
HAS_IMBLEARN = ensure("imblearn", "imbalanced-learn")
HAS_SHAP = ensure("shap")
print("optional libraries ->", {"xgboost": HAS_XGB, "imbalanced-learn": HAS_IMBLEARN, "shap": HAS_SHAP})

In [ ]:
# ---------------------------------------------------------------------
# 0.2  Imports
# ---------------------------------------------------------------------
import os
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    IsolationForest,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print(f"python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")

In [ ]:
# ---------------------------------------------------------------------
# 0.3  Central configuration + reproducibility
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class Config:
    """Every experiment knob in one auditable place."""

    random_state: int = 42
    test_size: float = 0.25                 # stratified hold-out fraction
    benign_label: str = "BENIGN"            # everything else is an attack (positive class)

    # Speed controls -- full data is used for the final fit, subsamples for search/CV.
    cv_sample_size: int = 150_000           # rows used for model-selection CV
    tune_sample_size: int = 150_000         # rows used for hyperparameter search
    cv_folds: int = 5
    search_iterations: int = 25             # RandomizedSearchCV candidates

    # Data-quality thresholds
    correlation_threshold: float = 0.95     # drop one of any pair above this
    missing_row_threshold: float = 0.5      # drop rows missing > 50% of features

    # Operating-point policy (see 3.4): the cost of missing an attack vs a false alarm
    cost_false_negative: float = 10.0
    cost_false_positive: float = 1.0
    max_acceptable_fpr: float = 0.01        # 1% of benign traffic may be escalated

    n_jobs: int = -1
    drive_dir: str = "/content/drive/MyDrive"
    dataset_files: tuple = ("Dataset1.csv", "Dataset2.csv", "Dataset3.csv")
    artefact_dir: str = "artifacts"


CFG = Config()

random.seed(CFG.random_state)
np.random.seed(CFG.random_state)
os.environ["PYTHONHASHSEED"] = str(CFG.random_state)
Path(CFG.artefact_dir).mkdir(exist_ok=True)

RESULTS: dict[str, dict] = {}   # populated through the notebook, summarised in Section 7

print("Configuration")
print("-" * 60)
for key, value in vars(CFG).items():
    print(f"{key:<24}: {value}")

In [ ]:
# ---------------------------------------------------------------------
# 0.4  Small, reusable helpers (kept together for readability)
# ---------------------------------------------------------------------
def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Strip/collapse whitespace in column names (CICIDS files have leading spaces)."""
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def find_col(df: pd.DataFrame, *candidates: str) -> str | None:
    """Return the first column matching any candidate, ignoring case/space/underscore.

    Makes feature engineering robust to the many CICIDS column-naming variants
    (e.g. 'Total Fwd Packets' vs 'Total Fwd Packet').
    """
    def key(name: str) -> str:
        return "".join(ch for ch in str(name).lower() if ch.isalnum())

    lookup = {key(c): c for c in df.columns}
    for cand in candidates:
        hit = lookup.get(key(cand))
        if hit is not None:
            return hit
    return None


def safe_div(numerator, denominator, fill: float = 0.0):
    """Element-wise division that never returns inf/NaN (guards zero-duration flows)."""
    num = pd.to_numeric(pd.Series(numerator).astype("float64"), errors="coerce")
    den = pd.to_numeric(pd.Series(denominator).astype("float64"), errors="coerce")
    out = num.to_numpy() / np.where(den.to_numpy() == 0, np.nan, den.to_numpy())
    return pd.Series(out, index=num.index).replace([np.inf, -np.inf], np.nan).fillna(fill)


def shrink_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to float32/int32 -- roughly halves memory on ~1M rows."""
    out = df.copy()
    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype("float32")
    for col in out.select_dtypes(include=["int64"]).columns:
        out[col] = pd.to_numeric(out[col], downcast="integer")
    return out


def memory_mb(df: pd.DataFrame) -> float:
    """Deep memory footprint of a DataFrame in MiB."""
    return df.memory_usage(deep=True).sum() / 1024 ** 2


def stratified_subsample(X: pd.DataFrame, y: pd.Series, n: int, seed: int = CFG.random_state):
    """Class-proportional subsample used to keep CV and search runtimes sane."""
    if n >= len(X):
        return X, y
    idx, _ = train_test_split(np.arange(len(X)), train_size=n, stratify=y, random_state=seed)
    return X.iloc[idx], y.iloc[idx]


def quartile_bands(values, labels) -> pd.Series:
    """Split values into equal-count bands.

    Ranks before cutting: raw flow features contain huge ties (many exact zeros), which makes
    plain `pd.qcut` collapse bin edges and raise. Ranking guarantees `len(labels)` non-empty bands.
    """
    ranked = pd.Series(np.asarray(values, dtype="float64")).rank(method="first")
    return pd.qcut(ranked, q=len(labels), labels=labels)


def show(title: str, frame: pd.DataFrame) -> None:
    """Print a titled table (works in Colab and in plain Jupyter)."""
    print(f"\n{title}\n" + "-" * max(len(title), 60))
    print(frame.to_string())


print("helpers ready:", [f.__name__ for f in
      (normalise_columns, find_col, safe_div, shrink_dtypes, memory_mb,
       stratified_subsample, quartile_bands, show)])

---
# 1. Data Understanding and Preparation  *(3 marks)*

**Plan**

1. **1.1 Load** the three CSV exports (Drive → local → simulated fallback).
2. **1.2 Merge & normalise** schema, record data provenance.
3. **1.3 Audit** structure: dtypes, missing values, infinities, duplicates, constant columns.
4. **1.4 Target** definition and class-imbalance analysis.
5. **1.5 EDA** — visualise imbalance, attack mix and separability of key features.
6. **1.6 Clean** — de-duplicate, fix ±∞, drop useless/redundant columns.
7. **1.7 Feature engineering** — domain-driven ratio/rate/asymmetry features.
8. **1.8 Split** — stratified train/test, with an explicit anti-leakage discussion.

In [ ]:
# ---------------------------------------------------------------------
# 1.1  Load the datasets: Google Drive -> local files -> simulated fallback
# ---------------------------------------------------------------------
def mount_drive() -> bool:
    """Mount Google Drive when running inside Colab. Returns True on success."""
    try:
        from google.colab import drive           # noqa: F401  (Colab-only import)
        drive.mount("/content/drive", force_remount=False)
        return True
    except Exception as exc:
        print(f"Google Drive not available ({type(exc).__name__}) -- trying local files.")
        return False


def resolve_paths() -> list[Path]:
    """Locate the three dataset CSVs in Drive, the CWD, or common Colab folders."""
    roots = [Path(CFG.drive_dir), Path("."), Path("/content"), Path("data"), Path("/content/sample_data")]
    found = []
    for name in CFG.dataset_files:
        for root in roots:
            candidate = root / name
            if candidate.exists():
                found.append(candidate)
                break
    return found


def simulate_cicids(n_rows: int = 120_000, seed: int = CFG.random_state) -> pd.DataFrame:
    """Generate a CICIDS-shaped SIMULATED dataset so the notebook can always run.

    This is a structural stand-in (same columns, same imbalance, same label vocabulary),
    NOT real traffic. Any result produced from it is clearly marked as simulated.
    """
    rng = np.random.default_rng(seed)
    mix = {  # label -> share of rows, mirroring CICIDS-2017 proportions
        "BENIGN": 0.78, "DoS Hulk": 0.075, "PortScan": 0.055, "DDoS": 0.04,
        "DoS GoldenEye": 0.012, "FTP-Patator": 0.008, "SSH-Patator": 0.006,
        "DoS slowloris": 0.006, "DoS Slowhttptest": 0.005, "Bot": 0.004,
        "Web Attack - Brute Force": 0.003, "Infiltration": 0.002, "Heartbleed": 0.001,
    }
    labels = rng.choice(list(mix), size=n_rows, p=np.array(list(mix.values())) / sum(mix.values()))
    is_attack = labels != "BENIGN"
    shift = np.where(is_attack, 1.0, 0.0)                      # attacks are shifted/tighter

    columns = [
        "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Fwd Packet Length Max",
        "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
        "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
        "Bwd Packet Length Std", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std",
        "Flow IAT Max", "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std",
        "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std",
        "Bwd IAT Max", "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags",
        "Bwd URG Flags", "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s",
        "Bwd Packets/s", "Min Packet Length", "Max Packet Length", "Packet Length Mean",
        "Packet Length Std", "Packet Length Variance", "FIN Flag Count", "SYN Flag Count",
        "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count",
        "ECE Flag Count", "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size",
        "Avg Bwd Segment Size", "Fwd Header Length.1", "Fwd Avg Bytes/Bulk",
        "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Fwd Packets",
        "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
        "Init_Win_bytes_forward", "Init_Win_bytes_backward", "act_data_pkt_fwd",
        "min_seg_size_forward", "Active Mean", "Active Std", "Active Max", "Active Min",
        "Idle Mean", "Idle Std", "Idle Max", "Idle Min",
    ]

    data = {}
    for i, col in enumerate(columns):
        base = rng.lognormal(mean=3.0 + (i % 5) * 0.6 - 1.1 * shift,
                             sigma=1.4 - 0.5 * shift,
                             size=n_rows)
        if "Flag" in col or "Ratio" in col or col.startswith(("Fwd Avg", "Bwd Avg")):
            base = rng.poisson(0.4 + 1.2 * shift, size=n_rows).astype(float)
        data[col] = np.round(base, 4)

    data["Destination Port"] = np.where(
        is_attack,
        rng.choice([21, 22, 80, 443, 445, 3389, 8080, 0], size=n_rows,
                   p=[.10, .10, .30, .18, .10, .10, .07, .05]),
        rng.choice([53, 80, 443, 389, 88, 445, 8080], size=n_rows,
                   p=[.14, .28, .38, .08, .05, .04, .03]),
    )
    data["Total Fwd Packets"] = rng.poisson(np.where(is_attack, 3, 11), n_rows) + 1
    data["Total Backward Packets"] = rng.poisson(np.where(is_attack, 1, 9), n_rows)
    data["Flow Duration"] = np.maximum(rng.lognormal(np.where(is_attack, 7.0, 11.0), 2.0, n_rows), 1)

    df = pd.DataFrame(data)
    df["Label"] = labels

    # Inject the real-world defects a data scientist must handle.
    df.loc[rng.choice(n_rows, size=int(0.004 * n_rows), replace=False), "Flow Bytes/s"] = np.inf
    df.loc[rng.choice(n_rows, size=int(0.003 * n_rows), replace=False), "Flow Packets/s"] = np.nan
    df = pd.concat([df, df.sample(int(0.02 * n_rows), random_state=seed)], ignore_index=True)
    df["Bwd PSH Flags"] = 0                                   # constant column, as in the real data
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)


DRIVE_MOUNTED = mount_drive()
paths = resolve_paths()
frames: list[pd.DataFrame] = []
DATA_SOURCE = "simulated"

if paths:
    for path in paths:
        part = pd.read_csv(path, low_memory=False)
        part["source_file"] = path.name
        frames.append(part)
        print(f"loaded {path}  ->  shape {part.shape}")
    DATA_SOURCE = "real"
else:
    print("\n" + "!" * 78)
    print("! Dataset1/2/3.csv were not found. Falling back to a SIMULATED CICIDS-shaped")
    print("! dataset so that every cell still runs. Re-run in Colab with the CSVs in")
    print(f"! {CFG.drive_dir} to reproduce these results on the real traffic.")
    print("!" * 78 + "\n")
    for i in range(3):
        part = simulate_cicids(seed=CFG.random_state + i)
        part["source_file"] = f"Dataset{i + 1}.csv (SIMULATED)"
        frames.append(part)
        print(f"generated Dataset{i + 1}.csv (SIMULATED) -> shape {part.shape}")

print(f"\nDATA_SOURCE = {DATA_SOURCE!r}")
display(frames[0].head())

In [ ]:
# ---------------------------------------------------------------------
# 1.2  Merge the three exports into one analysis frame
# ---------------------------------------------------------------------
frames = [normalise_columns(f) for f in frames]

common = set(frames[0].columns)
for f in frames[1:]:
    common &= set(f.columns)
print(f"columns per file : {[f.shape[1] for f in frames]}")
print(f"common columns   : {len(common)}")

raw = pd.concat([f[sorted(common)] for f in frames], ignore_index=True)
raw = shrink_dtypes(raw)

LABEL_COL = find_col(raw, "Label", "Attack", "Class") or "Label"
raw[LABEL_COL] = raw[LABEL_COL].astype(str).str.strip()

print(f"\ncombined shape   : {raw.shape}")
print(f"memory           : {memory_mb(raw):,.1f} MiB")
print(f"label column     : {LABEL_COL!r}")
show("Rows contributed per source file", raw["source_file"].value_counts().to_frame("rows"))

In [ ]:
# ---------------------------------------------------------------------
# 1.3  Structural audit -- what is actually wrong with this data?
# ---------------------------------------------------------------------
numeric_cols = raw.select_dtypes(include=[np.number]).columns.tolist()

audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(3),
    "n_unique": raw.nunique(dropna=False),
    "zeros_%": [(raw[c].eq(0).mean() * 100).round(2) if c in numeric_cols else np.nan
                for c in raw.columns],
})
audit["infinite"] = [int(np.isinf(raw[c].to_numpy(dtype="float64", na_value=np.nan)).sum())
                     if c in numeric_cols else 0 for c in raw.columns]

constant_cols = audit.index[audit["n_unique"] <= 1].tolist()
dup_rows = int(raw.duplicated().sum())
neg_cols = [c for c in numeric_cols if (raw[c] < 0).any()]

print(f"rows x cols                  : {raw.shape[0]:,} x {raw.shape[1]}")
print(f"numeric predictors           : {len(numeric_cols)}")
print(f"fully duplicated rows        : {dup_rows:,}  ({dup_rows / len(raw):.2%})")
print(f"columns containing +/-inf    : {int((audit['infinite'] > 0).sum())}")
print(f"columns with missing values   : {int((audit['missing'] > 0).sum())}")
print(f"constant (zero-variance) cols : {len(constant_cols)} -> {constant_cols}")
print(f"columns with negative values  : {len(neg_cols)} -> {neg_cols[:8]}")

show("Data-quality issues (columns needing attention)",
     audit[(audit["missing"] > 0) | (audit["infinite"] > 0) | (audit["n_unique"] <= 1)])
display(audit.head(20))

In [ ]:
# ---------------------------------------------------------------------
# 1.4  Target definition: binary (fraud/attack vs benign) + attack family
# ---------------------------------------------------------------------
label_counts = raw[LABEL_COL].value_counts()
label_table = pd.DataFrame({
    "rows": label_counts,
    "share_%": (label_counts / len(raw) * 100).round(4),
})
show("Label distribution (raw classes)", label_table)

raw["is_attack"] = (raw[LABEL_COL].str.upper() != CFG.benign_label.upper()).astype("int8")

n_pos = int(raw["is_attack"].sum())
n_neg = int(len(raw) - n_pos)
imbalance_ratio = n_neg / max(n_pos, 1)

print(f"\nbenign (0)          : {n_neg:,}  ({n_neg / len(raw):.2%})")
print(f"attack/fraud (1)    : {n_pos:,}  ({n_pos / len(raw):.2%})")
print(f"imbalance ratio     : {imbalance_ratio:.1f} benign per attack")
print(f"majority-class accuracy of a 'predict BENIGN always' model: "
      f"{max(n_neg, n_pos) / len(raw):.4%}  <-- why accuracy alone is useless here")
print(f"distinct attack families: {raw.loc[raw['is_attack'] == 1, LABEL_COL].nunique()}")

In [ ]:
# ---------------------------------------------------------------------
# 1.5  Exploratory data analysis
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

axes[0].bar(["Benign", "Attack/Fraud"], [n_neg, n_pos], color=["#2a9d8f", "#e63946"])
axes[0].set_title("Binary class balance")
axes[0].set_ylabel("flows")
for i, v in enumerate([n_neg, n_pos]):
    axes[0].text(i, v, f"{v:,}\n({v / len(raw):.1%})", ha="center", va="bottom", fontsize=9)

top_attacks = (raw.loc[raw["is_attack"] == 1, LABEL_COL]
               .value_counts().head(10).sort_values())
axes[1].barh(top_attacks.index, top_attacks.to_numpy(), color="#e76f51")
axes[1].set_title("Top attack families (log scale)")
axes[1].set_xscale("log")
axes[1].set_xlabel("flows (log)")
plt.tight_layout()
plt.show()

print("Takeaway: the positive class is a small minority AND is itself imbalanced across "
      "families -- rare families (Heartbleed, Infiltration) risk being ignored by a model "
      "tuned only on the aggregate metric.")

In [ ]:
# ---------------------------------------------------------------------
# 1.5b  Do the raw features separate the classes? (distributions + correlation)
# ---------------------------------------------------------------------
probe_names = ["Flow Duration", "Total Fwd Packets", "Flow IAT Mean", "Packet Length Mean",
               "Fwd Packet Length Mean", "Flow Bytes/s"]
probes = [c for c in (find_col(raw, n) for n in probe_names) if c is not None][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), probes):
    for cls, colour, name in [(0, "#2a9d8f", "benign"), (1, "#e63946", "attack")]:
        vals = raw.loc[raw["is_attack"] == cls, col]
        vals = pd.to_numeric(vals, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(vals) > 40_000:
            vals = vals.sample(40_000, random_state=CFG.random_state)
        ax.hist(np.log1p(vals.clip(lower=0)), bins=60, alpha=0.55, label=name,
                color=colour, density=True)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("log1p(value)")
    ax.legend(fontsize=7)
for ax in axes.ravel()[len(probes):]:
    ax.axis("off")
plt.suptitle("Class-conditional distributions of key flow features", y=1.01)
plt.tight_layout()
plt.show()

# Correlation structure on a sample (drives the redundancy pruning in 1.6)
sample = raw[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all")
sample = sample.sample(min(60_000, len(sample)), random_state=CFG.random_state).fillna(0)
corr = sample.corr(numeric_only=True).abs()

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.to_numpy(), cmap="magma", vmin=0, vmax=1)
ax.set_title(f"|Pearson correlation| of {corr.shape[0]} numeric features")
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, shrink=0.8, label="|r|")
plt.tight_layout()
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
redundant = [c for c in upper.columns if (upper[c] > CFG.correlation_threshold).any()]
print(f"{len(redundant)} features are >|{CFG.correlation_threshold}| correlated with an earlier "
      f"feature and will be dropped, e.g. {redundant[:10]}")

In [ ]:
# ---------------------------------------------------------------------
# 1.6  Cleaning pipeline (deterministic, order matters)
# ---------------------------------------------------------------------
def clean_flows(df: pd.DataFrame, drop_cols: list[str], label_col: str) -> pd.DataFrame:
    """Apply the documented cleaning steps and report the effect of each one.

    Steps: de-duplicate -> +/-inf to NaN -> drop empty rows -> median impute
    -> clip physically impossible negatives -> drop constant & redundant columns.
    """
    log: list[tuple[str, int, int]] = []
    out = df.copy()
    log.append(("input", *out.shape))

    out = out.drop_duplicates()
    log.append(("drop exact duplicates", *out.shape))

    num = out.select_dtypes(include=[np.number]).columns
    out[num] = out[num].replace([np.inf, -np.inf], np.nan)
    log.append(("+/-inf -> NaN", *out.shape))

    out = out.dropna(thresh=int((1 - CFG.missing_row_threshold) * len(num)), subset=num)
    log.append(("drop mostly-empty rows", *out.shape))

    out[num] = out[num].fillna(out[num].median(numeric_only=True))
    log.append(("median imputation", *out.shape))

    # Negative counters/durations are sensor artefacts, not signal.
    count_like = [c for c in num if any(k in c.lower() for k in
                  ("duration", "packets", "length", "bytes", "iat", "count", "size"))]
    out[count_like] = out[count_like].clip(lower=0)
    log.append(("clip negative counters", *out.shape))

    out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors="ignore")
    log.append(("drop constant + redundant cols", *out.shape))

    show("Cleaning audit trail",
         pd.DataFrame(log, columns=["step", "rows", "cols"]).set_index("step"))
    return out


# 'Fwd Header Length.1' is a known duplicate column in the CICIDS export.
to_drop = sorted(set(constant_cols) | set(redundant) | {find_col(raw, "Fwd Header Length.1") or ""} - {""})
to_drop = [c for c in to_drop if c not in {LABEL_COL, "is_attack", "source_file"}]

clean = clean_flows(raw, to_drop, LABEL_COL)
print(f"\nremoved {len(to_drop)} columns; kept {clean.shape[1]} "
      f"(incl. label, source_file, is_attack)")
print(f"memory after cleaning: {memory_mb(clean):,.1f} MiB")
print(f"remaining NaN: {int(clean.isna().sum().sum())} | "
      f"remaining inf: {int(np.isinf(clean.select_dtypes(include=[np.number]).to_numpy()).sum())}")

In [ ]:
# ---------------------------------------------------------------------
# 1.7  FEATURE ENGINEERING -- domain knowledge -> new predictors
# ---------------------------------------------------------------------
def engineer_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Add ratio / rate / asymmetry / service features that encode attacker behaviour.

    Rationale per family:
      * volume ratios   - scans and floods are extremely one-directional
      * per-packet size - floods use tiny uniform packets, exfiltration uses large ones
      * rate features   - DoS is defined by packets-per-second, brute force by retry cadence
      * timing shape    - burstiness (IAT std/mean) separates scripted from human traffic
      * flag density    - SYN/RST/FIN density is the signature of scans and half-open floods
      * service context - the *category* of the port generalises better than the port number
      * log transforms  - tame the heavy tails visible in 1.5b for the linear models
    """
    out = df.copy()
    created: list[str] = []

    def add(name: str, series) -> None:
        out[name] = pd.Series(series, index=out.index).astype("float32")
        created.append(name)

    c = {k: find_col(out, *v) for k, v in {
        "dur":      ("Flow Duration",),
        "fpkt":     ("Total Fwd Packets", "Total Fwd Packet"),
        "bpkt":     ("Total Backward Packets", "Total Bwd packets"),
        "flen":     ("Total Length of Fwd Packets", "Total Length of Fwd Packet"),
        "blen":     ("Total Length of Bwd Packets", "Total Length of Bwd Packet"),
        "fmean":    ("Fwd Packet Length Mean",),
        "bmean":    ("Bwd Packet Length Mean",),
        "fhdr":     ("Fwd Header Length",),
        "bhdr":     ("Bwd Header Length",),
        "iatmean":  ("Flow IAT Mean",),
        "iatstd":   ("Flow IAT Std",),
        "iatmax":   ("Flow IAT Max",),
        "iatmin":   ("Flow IAT Min",),
        "active":   ("Active Mean",),
        "idle":     ("Idle Mean",),
        "plmean":   ("Packet Length Mean",),
        "plstd":    ("Packet Length Std",),
        "initf":    ("Init_Win_bytes_forward",),
        "port":     ("Destination Port",),
    }.items()}

    # --- volume & directionality -------------------------------------
    if c["fpkt"] and c["bpkt"]:
        total_pkts = out[c["fpkt"]].astype("float64") + out[c["bpkt"]].astype("float64")
        add("fe_total_packets", total_pkts)
        add("fe_pkt_fwd_ratio", safe_div(out[c["fpkt"]], total_pkts))
        add("fe_pkt_direction_ratio", safe_div(out[c["fpkt"]], out[c["bpkt"]]))
        add("fe_is_unanswered", (out[c["bpkt"]] == 0).astype("float32"))   # scan signature
    if c["flen"] and c["blen"]:
        total_bytes = out[c["flen"]].astype("float64") + out[c["blen"]].astype("float64")
        add("fe_total_bytes", total_bytes)
        add("fe_byte_direction_ratio", safe_div(out[c["flen"]], out[c["blen"]]))
        add("fe_byte_fwd_ratio", safe_div(out[c["flen"]], total_bytes))

    # --- per-packet payload economics --------------------------------
    if c["flen"] and c["fpkt"]:
        add("fe_fwd_bytes_per_pkt", safe_div(out[c["flen"]], out[c["fpkt"]]))
    if c["blen"] and c["bpkt"]:
        add("fe_bwd_bytes_per_pkt", safe_div(out[c["blen"]], out[c["bpkt"]]))
    if "fe_total_bytes" in out and "fe_total_packets" in out:
        add("fe_avg_pkt_size", safe_div(out["fe_total_bytes"], out["fe_total_packets"]))
    if c["fmean"] and c["bmean"]:
        add("fe_pkt_size_asymmetry",
            safe_div(out[c["fmean"]].astype("float64") - out[c["bmean"]].astype("float64"),
                     out[c["fmean"]].astype("float64") + out[c["bmean"]].astype("float64")))

    # --- rates (recomputed defensively; the shipped Flow Bytes/s has inf) ---
    if c["dur"]:
        seconds = out[c["dur"]].astype("float64") / 1e6           # CICIDS duration is microseconds
        add("fe_duration_log", np.log1p(out[c["dur"]].clip(lower=0).astype("float64")))
        add("fe_is_microflow", (out[c["dur"]] < 1_000).astype("float32"))
        if "fe_total_bytes" in out:
            add("fe_bytes_per_sec", safe_div(out["fe_total_bytes"], seconds))
        if "fe_total_packets" in out:
            add("fe_pkts_per_sec", safe_div(out["fe_total_packets"], seconds))

    # --- header overhead (tiny-payload floods look like pure header) ---
    if c["fhdr"] and c["bhdr"] and "fe_total_bytes" in out:
        header = out[c["fhdr"]].astype("float64") + out[c["bhdr"]].astype("float64")
        add("fe_header_payload_ratio", safe_div(header, out["fe_total_bytes"]))

    # --- timing shape -------------------------------------------------
    if c["iatstd"] and c["iatmean"]:
        add("fe_iat_burstiness", safe_div(out[c["iatstd"]], out[c["iatmean"]]))
    if c["iatmax"] and c["iatmin"]:
        add("fe_iat_range", out[c["iatmax"]].astype("float64") - out[c["iatmin"]].astype("float64"))
    if c["active"] and c["idle"]:
        add("fe_active_idle_ratio", safe_div(out[c["active"]], out[c["idle"]]))
    if c["plstd"] and c["plmean"]:
        add("fe_pkt_len_cv", safe_div(out[c["plstd"]], out[c["plmean"]]))   # 0 => uniform => scripted

    # --- TCP flag density ---------------------------------------------
    flag_cols = [col for col in out.columns if "flag" in col.lower()
                 and out[col].dtype.kind in "iuf" and not col.startswith("fe_")]
    if flag_cols:
        flag_total = out[flag_cols].astype("float64").sum(axis=1)
        add("fe_flag_total", flag_total)
        if "fe_total_packets" in out:
            add("fe_flag_density", safe_div(flag_total, out["fe_total_packets"]))
    syn, rst, fin = (find_col(out, f"{f} Flag Count") for f in ("SYN", "RST", "FIN"))
    if syn and rst:
        add("fe_syn_rst_sum", out[syn].astype("float64") + out[rst].astype("float64"))
    if syn and fin:
        add("fe_half_open", ((out[syn] > 0) & (out[fin] == 0)).astype("float32"))

    # --- service context (generalises better than the raw port number) ---
    if c["port"]:
        port = pd.to_numeric(out[c["port"]], errors="coerce").fillna(-1)
        add("fe_port_is_system", (port <= 1023).astype("float32"))
        add("fe_port_is_registered", ((port > 1023) & (port <= 49151)).astype("float32"))
        add("fe_port_is_ephemeral", (port > 49151).astype("float32"))
        add("fe_port_is_web", port.isin([80, 8080, 8000, 8888]).astype("float32"))
        add("fe_port_is_encrypted", port.isin([443, 8443, 22, 993, 995]).astype("float32"))
        add("fe_port_is_remote_admin", port.isin([22, 23, 3389, 5900]).astype("float32"))
        add("fe_port_is_file_transfer", port.isin([20, 21, 445, 139, 2049]).astype("float32"))
        add("fe_port_is_infra", port.isin([53, 88, 389, 464, 123, 67, 68]).astype("float32"))

    # --- tail taming for the linear/distance-based baselines ------------
    for name in ("fe_bytes_per_sec", "fe_pkts_per_sec", "fe_total_bytes", "fe_total_packets"):
        if name in out:
            add(f"{name}_log", np.log1p(out[name].clip(lower=0).astype("float64")))

    out[created] = out[created].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out, created


featured, NEW_FEATURES = engineer_features(clean)
print(f"engineered {len(NEW_FEATURES)} new features "
      f"({clean.shape[1]} -> {featured.shape[1]} columns)")
show("Engineered features (describe)",
     featured[NEW_FEATURES].describe().T[["mean", "std", "min", "50%", "max"]])

In [ ]:
# ---------------------------------------------------------------------
# 1.7b  Are the engineered features actually informative?
# ---------------------------------------------------------------------
means = (featured.groupby("is_attack")[NEW_FEATURES].mean().T
         .rename(columns={0: "benign_mean", 1: "attack_mean"}))
pooled_std = featured[NEW_FEATURES].std().replace(0, np.nan)
means["std_gap"] = ((means["attack_mean"] - means["benign_mean"]).abs() / pooled_std).round(3)
show("Engineered features ranked by standardised class separation (top 15)",
     means.sort_values("std_gap", ascending=False).head(15))

top = means.sort_values("std_gap", ascending=False).head(12)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(top.index[::-1], top["std_gap"].to_numpy()[::-1], color="#457b9d")
ax.set_xlabel("|mean difference| / pooled std")
ax.set_title("Discriminative power of engineered features")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# 1.8  Feature matrix + stratified train/test split
# ---------------------------------------------------------------------
NON_FEATURES = {LABEL_COL, "is_attack", "source_file"}
FEATURES = [c for c in featured.columns
            if c not in NON_FEATURES and featured[c].dtype.kind in "iuf"]

# Anti-leakage decision: the raw Destination Port is a near-identifier that lets a tree
# memorise "port 3389 => attack" for this capture. We keep the engineered service-category
# features (which generalise) and drop the raw port from the model input.
RAW_PORT = find_col(featured, "Destination Port")
if RAW_PORT in FEATURES:
    FEATURES.remove(RAW_PORT)
    print(f"excluded {RAW_PORT!r} from the feature set (identifier-like -> memorisation risk)")

X = featured[FEATURES].astype("float32")
y = featured["is_attack"].astype("int8")
y_multi = featured[LABEL_COL].copy()

X_train, X_test, y_train, y_test, ymulti_train, ymulti_test = train_test_split(
    X, y, y_multi,
    test_size=CFG.test_size,
    stratify=y,
    random_state=CFG.random_state,
)

print(f"features used   : {len(FEATURES)}  ({len(NEW_FEATURES)} engineered)")
print(f"train           : {X_train.shape}  attack rate {y_train.mean():.4%}")
print(f"test            : {X_test.shape}  attack rate {y_test.mean():.4%}")
print(f"class weight for the positive class: {(y_train == 0).sum() / (y_train == 1).sum():.1f}")

print("""
Leakage controls applied
------------------------
1. Scaling / imputation happen INSIDE a Pipeline, so statistics are learned on training folds only.
2. Exact duplicate flows were removed BEFORE splitting, so an identical row cannot appear in both
   train and test and inflate the score.
3. The raw Destination Port (identifier-like) is excluded; only service categories are used.
4. No feature is derived from the label, and Section 5.3 adds a stricter capture-based hold-out.
""")

---
# 2. Model Selection  *(3 marks)*

**Selection strategy.** Rather than guessing, we screen a deliberately diverse shortlist with
*stratified 5-fold cross-validation* on a class-proportional subsample, and rank by
**average precision (PR-AUC)** — the metric that actually matters when the positive class is rare,
because it ignores the huge pool of true negatives that flatters ROC-AUC.

| Candidate | Why it is on the shortlist |
|---|---|
| `DummyClassifier` (stratified) | Sanity floor — any real model must beat it decisively. |
| `LogisticRegression` | Fast, monotone, fully interpretable coefficients; the *baseline to beat*. Needs scaling. |
| `DecisionTree` | Non-linear, human-readable rules an analyst can audit; high variance alone. |
| `RandomForest` | Bagged trees: strong on tabular data, robust to outliers/skew, gives importances. |
| `ExtraTrees` | More randomised splits → lower variance, very fast to fit. |
| `HistGradientBoosting` | Boosted histogram trees: usually SOTA on tabular data, handles NaN natively. |
| `XGBoost` *(if available)* | Industry standard with explicit `scale_pos_weight` for imbalance. |

**Imbalance handling.** We prefer `class_weight="balanced"` (cost-sensitive learning) over SMOTE
oversampling: it does not fabricate synthetic "attacks" that never occurred on the wire, is far
cheaper on ~1M rows, and leaves probability calibration easier to reason about. Section 2.3 tests
that choice empirically instead of asserting it.

In [ ]:
# ---------------------------------------------------------------------
# 2.1  Candidate models
# ---------------------------------------------------------------------
def build_candidates(pos_weight: float) -> dict[str, Pipeline]:
    """Return the shortlist. Linear/distance models get a scaler; trees do not need one."""
    models: dict[str, Pipeline] = {
        "Dummy (stratified)": Pipeline([
            ("clf", DummyClassifier(strategy="stratified", random_state=CFG.random_state)),
        ]),
        "LogisticRegression": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                       n_jobs=CFG.n_jobs, random_state=CFG.random_state)),
        ]),
        "DecisionTree": Pipeline([
            ("clf", DecisionTreeClassifier(max_depth=12, min_samples_leaf=20,
                                           class_weight="balanced",
                                           random_state=CFG.random_state)),
        ]),
        "RandomForest": Pipeline([
            ("clf", RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                           class_weight="balanced_subsample",
                                           n_jobs=CFG.n_jobs,
                                           random_state=CFG.random_state)),
        ]),
        "ExtraTrees": Pipeline([
            ("clf", ExtraTreesClassifier(n_estimators=200, min_samples_leaf=2,
                                         class_weight="balanced_subsample",
                                         n_jobs=CFG.n_jobs,
                                         random_state=CFG.random_state)),
        ]),
        "HistGradientBoosting": Pipeline([
            ("clf", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.1,
                                                   random_state=CFG.random_state)),
        ]),
    }
    if HAS_XGB:
        from xgboost import XGBClassifier
        models["XGBoost"] = Pipeline([
            ("clf", XGBClassifier(
                n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.9,
                colsample_bytree=0.9, reg_lambda=1.0, scale_pos_weight=pos_weight,
                tree_method="hist", eval_metric="aucpr", n_jobs=CFG.n_jobs,
                random_state=CFG.random_state,
            )),
        ])
    return models


POS_WEIGHT = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
CANDIDATES = build_candidates(POS_WEIGHT)
print(f"scale_pos_weight = {POS_WEIGHT:.2f}")
print(f"{len(CANDIDATES)} candidates: {list(CANDIDATES)}")

In [ ]:
# ---------------------------------------------------------------------
# 2.2  Stratified cross-validation screen (PR-AUC primary, F1 / ROC-AUC secondary)
# ---------------------------------------------------------------------
X_cv, y_cv = stratified_subsample(X_train, y_train, CFG.cv_sample_size)
cv = StratifiedKFold(n_splits=CFG.cv_folds, shuffle=True, random_state=CFG.random_state)
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc",
           "f1": "f1", "recall": "recall", "precision": "precision"}

print(f"CV on {len(X_cv):,} rows x {X_cv.shape[1]} features, "
      f"{CFG.cv_folds}-fold stratified (attack rate {y_cv.mean():.3%})\n")

rows = []
for name, model in CANDIDATES.items():
    started = time.perf_counter()
    scores = cross_validate(model, X_cv, y_cv, cv=cv, scoring=SCORING,
                            n_jobs=1, error_score="raise")
    rows.append({
        "model": name,
        "pr_auc": scores["test_pr_auc"].mean(),
        "pr_auc_sd": scores["test_pr_auc"].std(),
        "roc_auc": scores["test_roc_auc"].mean(),
        "f1": scores["test_f1"].mean(),
        "recall": scores["test_recall"].mean(),
        "precision": scores["test_precision"].mean(),
        "fit_s": scores["fit_time"].mean(),
        "score_s": scores["score_time"].mean(),
    })
    print(f"{name:<22} PR-AUC {rows[-1]['pr_auc']:.4f} (+/-{rows[-1]['pr_auc_sd']:.4f})  "
          f"F1 {rows[-1]['f1']:.4f}  fit {rows[-1]['fit_s']:.1f}s  "
          f"[{time.perf_counter() - started:.0f}s total]")

selection = (pd.DataFrame(rows).set_index("model")
             .sort_values("pr_auc", ascending=False))
show("Model-selection leaderboard (5-fold CV)", selection.round(4))
RESULTS["model_selection"] = selection.to_dict()

In [ ]:
# ---------------------------------------------------------------------
# 2.2b  Visual comparison: quality vs training cost
# ---------------------------------------------------------------------
plot_df = selection.drop(index=[i for i in selection.index if i.startswith("Dummy")])
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))

order = plot_df.sort_values("pr_auc")
axes[0].barh(order.index, order["pr_auc"], xerr=order["pr_auc_sd"],
             color="#264653", alpha=0.9)
axes[0].set_xlim(max(0.0, order["pr_auc"].min() - 0.05), 1.005)
axes[0].set_xlabel("PR-AUC (average precision)")
axes[0].set_title("Detection quality (higher is better)")
for i, v in enumerate(order["pr_auc"]):
    axes[0].text(v, i, f" {v:.4f}", va="center", fontsize=8)

axes[1].scatter(plot_df["fit_s"], plot_df["pr_auc"], s=90, color="#e76f51")
for name, r in plot_df.iterrows():
    axes[1].annotate(name, (r["fit_s"], r["pr_auc"]), fontsize=8,
                     xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("mean fit time per fold (s)")
axes[1].set_ylabel("PR-AUC")
axes[1].set_title("Quality vs training cost")
plt.tight_layout()
plt.show()

BEST_NAME = selection.index[0]
print(f"Selected architecture: {BEST_NAME}  (PR-AUC {selection.iloc[0]['pr_auc']:.4f})")
print(f"Best interpretable baseline: LogisticRegression "
      f"(PR-AUC {selection.loc['LogisticRegression', 'pr_auc']:.4f}) -- the gap quantifies what "
      f"non-linearity buys us.")

In [ ]:
# ---------------------------------------------------------------------
# 2.3  Testing the imbalance strategy: class weights vs SMOTE vs nothing
# ---------------------------------------------------------------------
X_imb, y_imb = stratified_subsample(X_train, y_train, min(60_000, CFG.cv_sample_size))
imb_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.random_state)

strategies: dict[str, object] = {
    "none (plain RF)": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=120, n_jobs=CFG.n_jobs,
                                       random_state=CFG.random_state))]),
    "class_weight=balanced_subsample": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=120, class_weight="balanced_subsample",
                                       n_jobs=CFG.n_jobs, random_state=CFG.random_state))]),
}
if HAS_IMBLEARN:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    strategies["SMOTE oversampling"] = ImbPipeline([
        ("smote", SMOTE(random_state=CFG.random_state, k_neighbors=5)),
        ("clf", RandomForestClassifier(n_estimators=120, n_jobs=CFG.n_jobs,
                                       random_state=CFG.random_state)),
    ])

imb_rows = []
for name, model in strategies.items():
    sc = cross_validate(model, X_imb, y_imb, cv=imb_cv,
                        scoring={"pr_auc": "average_precision", "recall": "recall",
                                 "precision": "precision", "f1": "f1"},
                        n_jobs=1, error_score="raise")
    imb_rows.append({"strategy": name, "pr_auc": sc["test_pr_auc"].mean(),
                     "recall": sc["test_recall"].mean(),
                     "precision": sc["test_precision"].mean(),
                     "f1": sc["test_f1"].mean(), "fit_s": sc["fit_time"].mean()})

show("Imbalance-handling comparison (3-fold CV)",
     pd.DataFrame(imb_rows).set_index("strategy").round(4))
print("""
Decision: keep cost-sensitive class weights.
SMOTE interpolates between minority flows, inventing traffic patterns that never traversed the
network; on strong tree ensembles it typically buys little PR-AUC while multiplying training cost
and degrading probability calibration -- which matters because Section 3.4 tunes a threshold on
those probabilities.
""")

---
# 3. Performance Measurement  *(3 marks)*

**Metric philosophy.** With ~80% benign traffic, a "predict everything benign" model already scores
high accuracy, so accuracy is reported only to expose that trap. The metrics we actually judge on:

| Metric | What it answers | Why it matters to XYZ |
|---|---|---|
| **Recall (TPR)** | Of all real attacks, how many did we catch? | Misses are breaches. |
| **Precision** | Of everything we alerted on, how much was real? | Low precision = analyst alert fatigue. |
| **PR-AUC** | Threshold-free quality on the rare class | Primary model-selection metric. |
| **ROC-AUC** | Ranking quality over all thresholds | Comparable across datasets, but optimistic under imbalance. |
| **F1 / MCC** | Balanced single number; MCC uses all four cells | MCC is the most honest single score for imbalance. |
| **Balanced accuracy** | Mean of per-class recall | Prevents the majority class from hiding failure. |
| **FPR** | Share of benign traffic escalated | Directly drives SOC workload / cost. |
| **Expected cost** | 10 × FN + 1 × FP | Converts the confusion matrix into money. |

We also **never** judge the model at the default 0.5 cut-off only — Section 3.4 selects the operating
threshold from the precision–recall trade-off, using the training data, then reports it on the test set.

In [ ]:
# ---------------------------------------------------------------------
# 3.1  Fit the selected architecture on the FULL training set
# ---------------------------------------------------------------------
baseline_model = CANDIDATES[BEST_NAME]

t0 = time.perf_counter()
baseline_model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
proba_test = baseline_model.predict_proba(X_test)[:, 1]
score_seconds = time.perf_counter() - t0
pred_test = (proba_test >= 0.5).astype(int)

print(f"model      : {BEST_NAME}")
print(f"trained on : {X_train.shape[0]:,} flows x {X_train.shape[1]} features in {fit_seconds:.1f}s")
print(f"scored     : {X_test.shape[0]:,} flows in {score_seconds:.2f}s "
      f"({X_test.shape[0] / max(score_seconds, 1e-9):,.0f} flows/s)")

In [ ]:
# ---------------------------------------------------------------------
# 3.2  Metric suite
# ---------------------------------------------------------------------
def evaluate(y_true, proba, threshold: float = 0.5, name: str = "model") -> dict:
    """Compute the full metric suite at a given decision threshold."""
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": name,
        "threshold": round(float(threshold), 4),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "fpr": fp / max(fp + tn, 1),
        "fnr": fn / max(fn + tp, 1),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "expected_cost": CFG.cost_false_negative * fn + CFG.cost_false_positive * fp,
    }


def threshold_sweep(y_true, proba, thresholds) -> pd.DataFrame:
    """Vectorised metric curve over many thresholds.

    Calling `evaluate` inside a 99-point loop would rescan ~800k rows a dozen times per point.
    Sorting once and using `searchsorted` gives the exact same confusion-matrix counts in one pass,
    turning a multi-minute sweep into milliseconds.
    """
    truth = np.asarray(y_true).astype(int)
    scores = np.asarray(proba, dtype="float64")
    pos = np.sort(scores[truth == 1])
    neg = np.sort(scores[truth == 0])
    n_pos, n_neg = len(pos), len(neg)

    grid_arr = np.asarray(thresholds, dtype="float64")
    tp = n_pos - np.searchsorted(pos, grid_arr, side="left")   # predicted 1 == score >= threshold
    fp = n_neg - np.searchsorted(neg, grid_arr, side="left")
    fn, tn = n_pos - tp, n_neg - fp

    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / max(n_pos, 1)
    specificity = tn / max(n_neg, 1)
    return pd.DataFrame({
        "threshold": grid_arr,
        "precision": precision,
        "recall": recall,
        "f1": np.where(precision + recall > 0, 2 * precision * recall,
                       np.nan) / np.maximum(precision + recall, 1e-12),
        "balanced_acc": (recall + specificity) / 2,
        "fpr": fp / max(n_neg, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "expected_cost": CFG.cost_false_negative * fn + CFG.cost_false_positive * fp,
    }).fillna({"f1": 0.0})


baseline_metrics = evaluate(y_test, proba_test, 0.5, f"{BEST_NAME} @0.50")
majority = evaluate(y_test, np.zeros(len(y_test)), 0.5, "always-BENIGN")

show("Test-set performance", pd.DataFrame([majority, baseline_metrics]).set_index("model").T)

print("\nPer-class report")
print(classification_report(y_test, pred_test, target_names=["BENIGN", "ATTACK"], digits=4))
print(f"Note the trap: the useless always-BENIGN model scores {majority['accuracy']:.2%} accuracy "
      f"but {majority['recall']:.0%} recall and MCC {majority['mcc']:.2f}.")

In [ ]:
# ---------------------------------------------------------------------
# 3.3  Confusion matrix, ROC and precision-recall curves
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test, display_labels=["BENIGN", "ATTACK"],
    cmap="Blues", colorbar=False, values_format=",d", ax=axes[0])
axes[0].set_title(f"Confusion matrix @0.50\n{BEST_NAME}")
axes[0].grid(False)

fpr_c, tpr_c, _ = roc_curve(y_test, proba_test)
axes[1].plot(fpr_c, tpr_c, color="#e63946", lw=2,
             label=f"AUC = {baseline_metrics['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="random")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")

prec_c, rec_c, _ = precision_recall_curve(y_test, proba_test)
axes[2].plot(rec_c, prec_c, color="#2a9d8f", lw=2,
             label=f"AP = {baseline_metrics['pr_auc']:.4f}")
axes[2].axhline(y_test.mean(), ls="--", c="k", lw=1,
                label=f"baseline = {y_test.mean():.3f}")
axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision")
axes[2].set_title("Precision-Recall curve"); axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# 3.4  Choosing the operating threshold from business cost (not from 0.5)
# ---------------------------------------------------------------------
proba_train = baseline_model.predict_proba(X_train)[:, 1]     # threshold chosen on TRAIN only

grid = np.unique(np.round(np.linspace(0.01, 0.99, 99), 4))
sweep = threshold_sweep(y_train, proba_train, grid)

cost_optimal = float(sweep.loc[sweep["expected_cost"].idxmin(), "threshold"])
f1_optimal = float(sweep.loc[sweep["f1"].idxmax(), "threshold"])
feasible = sweep[sweep["fpr"] <= CFG.max_acceptable_fpr]
sla_threshold = float(feasible.loc[feasible["recall"].idxmax(), "threshold"]) if len(feasible) else 0.5

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
axes[0].plot(sweep["threshold"], sweep["precision"], label="precision", color="#2a9d8f")
axes[0].plot(sweep["threshold"], sweep["recall"], label="recall", color="#e63946")
axes[0].plot(sweep["threshold"], sweep["f1"], label="F1", color="#264653", ls="--")
axes[0].axvline(cost_optimal, c="#e9c46a", lw=2, label=f"min cost = {cost_optimal:.2f}")
axes[0].set_xlabel("decision threshold"); axes[0].set_title("Precision / recall trade-off (train)")
axes[0].legend(fontsize=8)

axes[1].plot(sweep["threshold"], sweep["expected_cost"], color="#e76f51", lw=2)
axes[1].axvline(cost_optimal, c="#e9c46a", lw=2)
axes[1].set_xlabel("decision threshold")
axes[1].set_ylabel(f"{CFG.cost_false_negative:.0f}xFN + {CFG.cost_false_positive:.0f}xFP")
axes[1].set_title("Expected operational cost (train)")
plt.tight_layout()
plt.show()

comparison = pd.DataFrame([
    evaluate(y_test, proba_test, 0.5, "default 0.50"),
    evaluate(y_test, proba_test, f1_optimal, f"max-F1 {f1_optimal:.2f}"),
    evaluate(y_test, proba_test, cost_optimal, f"min-cost {cost_optimal:.2f}"),
    evaluate(y_test, proba_test, sla_threshold, f"FPR<={CFG.max_acceptable_fpr:.0%} {sla_threshold:.2f}"),
]).set_index("model")
show("Test-set metrics at four candidate operating points", comparison.T)

OPERATING_THRESHOLD = cost_optimal
print(f"\nAdopted operating threshold = {OPERATING_THRESHOLD:.3f} "
      f"(cost-optimal, selected on TRAIN, reported on TEST -- no test-set tuning).")
RESULTS["baseline_test"] = baseline_metrics

---
# 4. Hyperparameter Tuning  *(5 marks)*

**Method.** `RandomizedSearchCV` over a deliberately wide distribution, scored with
**average precision** under stratified 3-fold CV on a class-proportional subsample.

*Why randomised rather than exhaustive grid search?* With 6–7 interacting hyperparameters a grid
explodes combinatorially, and most parameters have a flat response surface; random search reaches a
comparable optimum with a fixed, predictable budget
([Bergstra & Bengio, 2012](https://www.jmlr.org/papers/v13/bergstra12a.html)). We keep the budget
explicit in `CFG.search_iterations` so the run is reproducible.

**Guard-rails**
* the search only ever sees **training** data — the test set stays untouched until 4.4;
* every candidate is scored with the *same* stratified folds (`random_state` fixed);
* the winner is refit on the full training set and compared against the untuned baseline;
* the operating threshold is re-derived for the tuned model, again on training data only.

In [ ]:
# ---------------------------------------------------------------------
# 4.1  Search space for the selected architecture
# ---------------------------------------------------------------------
from scipy.stats import loguniform, randint, uniform


def build_search_space(name: str):
    """Return (estimator, param_distributions) for the architecture chosen in Section 2."""
    if name == "XGBoost" and HAS_XGB:
        from xgboost import XGBClassifier
        est = Pipeline([("clf", XGBClassifier(
            tree_method="hist", eval_metric="aucpr", n_jobs=CFG.n_jobs,
            random_state=CFG.random_state, scale_pos_weight=POS_WEIGHT))])
        space = {
            "clf__n_estimators": randint(150, 700),
            "clf__max_depth": randint(3, 12),
            "clf__learning_rate": loguniform(0.01, 0.3),
            "clf__subsample": uniform(0.6, 0.4),
            "clf__colsample_bytree": uniform(0.5, 0.5),
            "clf__min_child_weight": randint(1, 12),
            "clf__reg_lambda": loguniform(0.1, 20.0),
            "clf__gamma": uniform(0.0, 5.0),
        }
    elif name == "HistGradientBoosting":
        est = Pipeline([("clf", HistGradientBoostingClassifier(
            random_state=CFG.random_state, early_stopping=False))])
        space = {
            "clf__max_iter": randint(150, 700),
            "clf__learning_rate": loguniform(0.02, 0.3),
            "clf__max_leaf_nodes": randint(15, 128),
            "clf__min_samples_leaf": randint(5, 80),
            "clf__l2_regularization": loguniform(1e-4, 10.0),
            "clf__max_bins": randint(64, 256),
        }
    elif name in {"RandomForest", "ExtraTrees", "Dummy (stratified)", "DecisionTree",
                  "LogisticRegression"}:
        cls = ExtraTreesClassifier if name == "ExtraTrees" else RandomForestClassifier
        est = Pipeline([("clf", cls(n_jobs=CFG.n_jobs, random_state=CFG.random_state))])
        space = {
            "clf__n_estimators": randint(150, 600),
            "clf__max_depth": [None, 12, 20, 30, 40],
            "clf__min_samples_split": randint(2, 20),
            "clf__min_samples_leaf": randint(1, 12),
            "clf__max_features": ["sqrt", "log2", 0.3, 0.5],
            "clf__class_weight": [None, "balanced", "balanced_subsample"],
            "clf__bootstrap": [True, False],
        }
    else:
        raise ValueError(f"no search space defined for {name!r}")
    return est, space


TUNE_ARCH = BEST_NAME if BEST_NAME in {"XGBoost", "HistGradientBoosting",
                                       "RandomForest", "ExtraTrees"} else "RandomForest"
estimator, search_space = build_search_space(TUNE_ARCH)
print(f"tuning architecture : {TUNE_ARCH}")
print(f"hyperparameters     : {len(search_space)}")
for key, dist in search_space.items():
    label = dist if isinstance(dist, list) else type(dist).__name__
    print(f"  {key:<32} {label}")

In [ ]:
# ---------------------------------------------------------------------
# 4.2  Run the randomised search
# ---------------------------------------------------------------------
X_tune, y_tune = stratified_subsample(X_train, y_train, CFG.tune_sample_size)
tune_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.random_state)

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=search_space,
    n_iter=CFG.search_iterations,
    scoring="average_precision",          # PR-AUC: the right target under imbalance
    cv=tune_cv,
    n_jobs=CFG.n_jobs,
    refit=True,
    random_state=CFG.random_state,
    return_train_score=True,
    verbose=1,
)

print(f"searching {CFG.search_iterations} candidates x 3 folds "
      f"= {CFG.search_iterations * 3} fits on {len(X_tune):,} rows ...")
t0 = time.perf_counter()
search.fit(X_tune, y_tune)
search_seconds = time.perf_counter() - t0

print(f"\ncompleted in {search_seconds / 60:.1f} min")
print(f"best CV PR-AUC : {search.best_score_:.5f}")
print("best parameters:")
for key, value in sorted(search.best_params_.items()):
    print(f"  {key:<32} {value}")

In [ ]:
# ---------------------------------------------------------------------
# 4.3  What did the search learn? (top candidates + overfitting check)
# ---------------------------------------------------------------------
cv_results = pd.DataFrame(search.cv_results_)
param_cols = [c for c in cv_results.columns if c.startswith("param_")]
leaderboard = (cv_results[["rank_test_score", "mean_test_score", "std_test_score",
                           "mean_train_score", "mean_fit_time"] + param_cols]
               .sort_values("rank_test_score")
               .head(10)
               .set_index("rank_test_score"))
leaderboard["overfit_gap"] = (leaderboard["mean_train_score"]
                              - leaderboard["mean_test_score"]).round(5)
leaderboard.columns = [c.replace("param_clf__", "") for c in leaderboard.columns]
show("Top-10 hyperparameter candidates", leaderboard.round(5))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
ranked = cv_results.sort_values("rank_test_score")
axes[0].errorbar(range(1, len(ranked) + 1), ranked["mean_test_score"],
                 yerr=ranked["std_test_score"], fmt="o-", ms=4, lw=1, color="#264653")
axes[0].axhline(search.best_score_, ls="--", c="#e63946",
                label=f"best = {search.best_score_:.5f}")
axes[0].set_xlabel("candidate (ranked)"); axes[0].set_ylabel("CV PR-AUC")
axes[0].set_title("Search landscape: candidate quality"); axes[0].legend(fontsize=8)

axes[1].scatter(cv_results["mean_train_score"], cv_results["mean_test_score"],
                c=cv_results["mean_fit_time"], cmap="viridis", s=60)
lims = [min(cv_results["mean_test_score"].min(), cv_results["mean_train_score"].min()) - 0.002, 1.001]
axes[1].plot(lims, lims, "k--", lw=1)
axes[1].set_xlabel("mean TRAIN PR-AUC"); axes[1].set_ylabel("mean CV PR-AUC")
axes[1].set_title("Generalisation gap (colour = fit time)")
plt.colorbar(axes[1].collections[0], ax=axes[1], label="fit time (s)")
plt.tight_layout()
plt.show()

# Which hyperparameter mattered most? (marginal effect on CV score)
impact = []
for col in param_cols:
    series = cv_results[col].astype(str)
    if series.nunique() > 1:
        grouped = cv_results.groupby(series)["mean_test_score"].mean()
        impact.append({"hyperparameter": col.replace("param_clf__", ""),
                       "spread_in_cv_pr_auc": round(float(grouped.max() - grouped.min()), 5),
                       "best_value": grouped.idxmax()})
show("Marginal sensitivity of CV PR-AUC to each hyperparameter",
     pd.DataFrame(impact).sort_values("spread_in_cv_pr_auc", ascending=False)
     .set_index("hyperparameter"))

In [ ]:
# ---------------------------------------------------------------------
# 4.4  Refit the winner on the full training set and compare like-for-like
# ---------------------------------------------------------------------
tuned_model = search.best_estimator_
t0 = time.perf_counter()
tuned_model.fit(X_train, y_train)                 # refit on ALL training rows
tuned_fit_seconds = time.perf_counter() - t0

proba_tuned = tuned_model.predict_proba(X_test)[:, 1]
proba_tuned_train = tuned_model.predict_proba(X_train)[:, 1]

tuned_sweep = threshold_sweep(y_train, proba_tuned_train, grid)
tuned_threshold = float(tuned_sweep.loc[tuned_sweep["expected_cost"].idxmin(), "threshold"])

before = evaluate(y_test, proba_test, OPERATING_THRESHOLD, "baseline (default HPs)")
after = evaluate(y_test, proba_tuned, tuned_threshold, "tuned (randomised search)")

delta = pd.DataFrame([before, after]).set_index("model").T
delta["change"] = delta.iloc[:, 1] - delta.iloc[:, 0]
show("Untuned vs tuned -- test set, each at its own cost-optimal threshold", delta)

metrics_to_plot = ["recall", "precision", "f1", "mcc", "pr_auc", "roc_auc", "balanced_acc"]
xs = np.arange(len(metrics_to_plot))
fig, ax = plt.subplots(figsize=(10, 4.3))
ax.bar(xs - 0.2, [before[m] for m in metrics_to_plot], 0.4, label="untuned", color="#8d99ae")
ax.bar(xs + 0.2, [after[m] for m in metrics_to_plot], 0.4, label="tuned", color="#2a9d8f")
ax.set_xticks(xs); ax.set_xticklabels(metrics_to_plot, rotation=20)
ax.set_ylim(min(min(before[m] for m in metrics_to_plot),
                min(after[m] for m in metrics_to_plot)) - 0.02, 1.005)
ax.set_title("Effect of hyperparameter tuning on the test set"); ax.legend()
plt.tight_layout()
plt.show()

print(f"missed attacks (FN) : {before['fn']:,} -> {after['fn']:,}")
print(f"false alarms  (FP)  : {before['fp']:,} -> {after['fp']:,}")
print(f"expected cost       : {before['expected_cost']:,.0f} -> {after['expected_cost']:,.0f} "
      f"({(after['expected_cost'] - before['expected_cost']) / max(before['expected_cost'], 1):+.2%})")
print(f"training time       : {fit_seconds:.1f}s -> {tuned_fit_seconds:.1f}s")

FINAL_MODEL, FINAL_PROBA, FINAL_THRESHOLD = ((tuned_model, proba_tuned, tuned_threshold)
    if after["pr_auc"] >= before["pr_auc"] else (baseline_model, proba_test, OPERATING_THRESHOLD))
FINAL_NAME = "tuned" if FINAL_MODEL is tuned_model else "untuned baseline"
FINAL_PRED = (FINAL_PROBA >= FINAL_THRESHOLD).astype(int)
RESULTS["tuned_test"] = after
print(f"\nPRODUCTION CANDIDATE: {FINAL_NAME} {TUNE_ARCH} @ threshold {FINAL_THRESHOLD:.3f}")

---
# 5. Extra Features and Considerations  *(3 marks)*

Five additions that turn a notebook model into something a SOC could actually run:

1. **5.1 Multi-class attack attribution** — *which* attack is it? Triage needs the family, not just a flag.
2. **5.2 Explainability** — impurity + permutation importance, so an analyst can justify a block.
3. **5.3 Capture-based generalisation test** — train on captures 1–2, test on capture 3 (distribution shift).
4. **5.4 Zero-day / novelty detection** — an unsupervised `IsolationForest` for attack families the
   supervised model has never seen (held out of training entirely).
5. **5.5 Deployment readiness** — latency, throughput, model size, persistence and a single-flow
   scoring API, plus a monitoring/retraining plan.

In [ ]:
# ---------------------------------------------------------------------
# 5.1  Multi-class attack attribution (which family is it?)
# ---------------------------------------------------------------------
# Rare families cannot be learned or evaluated reliably; group them explicitly rather than silently.
MIN_CLASS_ROWS = 200
family_counts = ymulti_train.value_counts()
keep_families = family_counts[family_counts >= MIN_CLASS_ROWS].index
ymulti_train_g = ymulti_train.where(ymulti_train.isin(keep_families), "OTHER-RARE")
ymulti_test_g = ymulti_test.where(ymulti_test.isin(keep_families), "OTHER-RARE")

print(f"families kept as their own class : {len(keep_families)}")
print(f"families folded into OTHER-RARE  : {int((family_counts < MIN_CLASS_ROWS).sum())} "
      f"-> {family_counts[family_counts < MIN_CLASS_ROWS].index.tolist()}")

# Stratification needs >= 2 rows per class; drop any class that still cannot be split.
grouped_counts = ymulti_train_g.value_counts()
trainable = grouped_counts[grouped_counts >= 2].index
mc_mask = ymulti_train_g.isin(trainable).to_numpy()
if (~mc_mask).any():
    print(f"dropped {int((~mc_mask).sum())} rows from classes with <2 examples (cannot stratify)")

X_mc, y_mc = stratified_subsample(X_train[mc_mask], ymulti_train_g[mc_mask],
                                  min(120_000, int(mc_mask.sum())))
multiclass_model = RandomForestClassifier(
    n_estimators=250, min_samples_leaf=2, class_weight="balanced_subsample",
    n_jobs=CFG.n_jobs, random_state=CFG.random_state)
multiclass_model.fit(X_mc, y_mc)

mc_pred = multiclass_model.predict(X_test)
print(f"\nmulti-class accuracy        : {accuracy_score(ymulti_test_g, mc_pred):.4f}")
print(f"macro F1 (all families equal): "
      f"{f1_score(ymulti_test_g, mc_pred, average='macro', zero_division=0):.4f}")
print(f"weighted F1                 : "
      f"{f1_score(ymulti_test_g, mc_pred, average='weighted', zero_division=0):.4f}")
print("\nPer-family report (macro F1 exposes the rare families the flat metric hides)")
print(classification_report(ymulti_test_g, mc_pred, digits=3, zero_division=0))

In [ ]:
# ---------------------------------------------------------------------
# 5.1b  Row-normalised confusion matrix -- where does attribution get confused?
# ---------------------------------------------------------------------
families = sorted(set(ymulti_test_g.unique()) | set(pd.Series(mc_pred).unique()))
cm = confusion_matrix(ymulti_test_g, mc_pred, labels=families)
cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(min(1.0 * len(families) + 3, 13),
                                min(0.9 * len(families) + 2.5, 11)))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(families))); ax.set_xticklabels(families, rotation=90, fontsize=8)
ax.set_yticks(range(len(families))); ax.set_yticklabels(families, fontsize=8)
ax.set_xlabel("predicted family"); ax.set_ylabel("true family")
ax.set_title("Attack attribution: row-normalised confusion matrix")
ax.grid(False)
for i in range(len(families)):
    for j in range(len(families)):
        if cm_norm[i, j] >= 0.01:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, shrink=0.75, label="share of true family")
plt.tight_layout()
plt.show()

per_family = pd.DataFrame({
    "support": cm.sum(axis=1),
    "recall": cm.diagonal() / np.clip(cm.sum(axis=1), 1, None),
}, index=families).sort_values("recall")
show("Weakest-detected families (fairness across attack types, see Section 6.3)",
     per_family.head(8).round(4))

In [ ]:
# ---------------------------------------------------------------------
# 5.2  Explainability: impurity importance + permutation importance
# ---------------------------------------------------------------------
def get_importances(pipeline, feature_names: list[str]) -> pd.Series | None:
    """Extract built-in feature importances from the final step of a Pipeline."""
    est = pipeline[-1] if isinstance(pipeline, Pipeline) else pipeline
    if hasattr(est, "feature_importances_"):
        return pd.Series(est.feature_importances_, index=feature_names)
    if hasattr(est, "coef_"):
        return pd.Series(np.abs(est.coef_).ravel(), index=feature_names)
    return None


impurity = get_importances(FINAL_MODEL, FEATURES)

# Permutation importance is model-agnostic and measures the metric drop when a feature is shuffled,
# so it is not fooled by high-cardinality features the way impurity importance is.
X_perm, y_perm = stratified_subsample(X_test, y_test, min(20_000, len(X_test)))
perm = permutation_importance(FINAL_MODEL, X_perm, y_perm, n_repeats=5,
                              scoring="average_precision",
                              random_state=CFG.random_state, n_jobs=CFG.n_jobs)
perm_series = pd.Series(perm.importances_mean, index=FEATURES)

importance_table = pd.DataFrame({
    "permutation_drop_in_pr_auc": perm_series,
    "permutation_sd": pd.Series(perm.importances_std, index=FEATURES),
    "impurity_importance": impurity if impurity is not None else np.nan,
    "engineered": [f in NEW_FEATURES for f in FEATURES],
}).sort_values("permutation_drop_in_pr_auc", ascending=False)
show("Top-20 features by permutation importance", importance_table.head(20).round(5))

top20 = importance_table.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.barh(top20.index, top20["permutation_drop_in_pr_auc"],
        xerr=top20["permutation_sd"],
        color=["#e76f51" if e else "#457b9d" for e in top20["engineered"]])
ax.set_xlabel("drop in PR-AUC when the feature is shuffled")
ax.set_title("Feature importance (orange = engineered in Section 1.7)")
plt.tight_layout()
plt.show()

engineered_share = importance_table.head(20)["engineered"].mean()
print(f"{engineered_share:.0%} of the top-20 most important features were engineered in Section 1.7 "
      f"-- direct evidence that the feature engineering paid off.")

if HAS_SHAP and hasattr(FINAL_MODEL[-1], "feature_importances_"):
    try:
        import shap
        sample = X_test.sample(min(2_000, len(X_test)), random_state=CFG.random_state)
        explainer = shap.TreeExplainer(FINAL_MODEL[-1])
        shap_values = explainer.shap_values(sample, check_additivity=False)
        values = shap_values[1] if isinstance(shap_values, list) else shap_values
        if values.ndim == 3:
            values = values[:, :, 1]
        shap.summary_plot(values, sample, feature_names=FEATURES, max_display=15, show=True)
    except Exception as exc:
        print(f"SHAP summary skipped ({type(exc).__name__}: {exc})")

In [ ]:
# ---------------------------------------------------------------------
# 5.3  Harder generalisation test: hold out an entire capture file
# ---------------------------------------------------------------------
# A random split lets flows from the same capture session sit on both sides. Training on captures
# 1-2 and testing on capture 3 approximates deployment: a model trained on the past, run on the future.
sources = sorted(featured["source_file"].unique())
if len(sources) >= 2:
    holdout = sources[-1]
    train_mask = featured["source_file"] != holdout
    Xg_train, yg_train = X[train_mask.to_numpy()], y[train_mask.to_numpy()]
    Xg_test, yg_test = X[~train_mask.to_numpy()], y[~train_mask.to_numpy()]

    print(f"train captures : {sources[:-1]}  ({len(Xg_train):,} flows, "
          f"attack rate {yg_train.mean():.2%})")
    print(f"held-out       : {holdout}  ({len(Xg_test):,} flows, "
          f"attack rate {yg_test.mean():.2%})")

    if yg_train.nunique() > 1 and yg_test.nunique() > 1:
        cross_capture = RandomForestClassifier(
            n_estimators=200, min_samples_leaf=2, class_weight="balanced_subsample",
            n_jobs=CFG.n_jobs, random_state=CFG.random_state).fit(Xg_train, yg_train)
        proba_g = cross_capture.predict_proba(Xg_test)[:, 1]
        cross_metrics = evaluate(yg_test, proba_g, FINAL_THRESHOLD, f"hold out {holdout}")
        show("Random split vs capture-based split (generalisation stress test)",
             pd.DataFrame([RESULTS["tuned_test"], cross_metrics]).set_index("model").T)
        print("""
Interpretation: a large drop here (and not in the random split) is the signature of
capture-specific memorisation -- the model latching onto artefacts of one recording session
rather than attack behaviour. This is the number to trust when forecasting live performance.
""")
    else:
        print("held-out capture has a single class -- cross-capture test not applicable")
else:
    print("only one source file present -- cross-capture test skipped")

In [ ]:
# ---------------------------------------------------------------------
# 5.4  Zero-day readiness: unsupervised novelty detection
# ---------------------------------------------------------------------
# The detector below is fitted on BENIGN traffic ONLY -- it has never seen a single labelled attack,
# so every attack family is a genuine "zero day" from its point of view. We measure how much of one
# specific family it still flags, which is the coverage we could expect against a novel threat.
attack_families = ymulti_train.loc[y_train == 1].value_counts()
if len(attack_families) >= 2:
    zero_day = attack_families.index[min(1, len(attack_families) - 1)]  # a mid-frequency family
    benign_train = X_train[(y_train == 0).to_numpy()]
    benign_fit = benign_train.sample(min(80_000, len(benign_train)), random_state=CFG.random_state)

    iso = IsolationForest(n_estimators=250, contamination=0.05, max_samples=min(50_000, len(benign_fit)),
                          n_jobs=CFG.n_jobs, random_state=CFG.random_state).fit(benign_fit)

    unseen = X_test[(ymulti_test == zero_day).to_numpy()]
    benign_test = X_test[(y_test == 0).to_numpy()]
    benign_eval = benign_test.sample(min(50_000, len(benign_test)), random_state=CFG.random_state)

    if len(unseen) > 0:
        detect_rate = float((iso.predict(unseen) == -1).mean())
        false_alarm = float((iso.predict(benign_eval) == -1).mean())
        print(f"simulated zero-day family : {zero_day!r} ({len(unseen):,} test flows)")
        print(f"detected as anomalous     : {detect_rate:.2%}  (recall on a threat never trained on)")
        print(f"benign flagged anomalous  : {false_alarm:.2%}  (unsupervised false-alarm cost)")

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(iso.score_samples(benign_eval), bins=70, alpha=0.6, density=True,
                label="benign", color="#2a9d8f")
        ax.hist(iso.score_samples(unseen), bins=70, alpha=0.6, density=True,
                label=f"unseen: {zero_day}", color="#e63946")
        ax.set_xlabel("IsolationForest anomaly score (lower = more anomalous)")
        ax.set_ylabel("density"); ax.set_title("Novelty detection for an unseen attack family")
        ax.legend()
        plt.tight_layout()
        plt.show()
        print("""
Design conclusion: run the two models in parallel. The supervised classifier gives high-precision
alerts on KNOWN attacks; the anomaly detector provides low-precision but non-zero coverage of
genuinely novel behaviour, routed to a separate, lower-priority analyst queue.
""")
    else:
        print(f"family {zero_day!r} absent from the test set -- zero-day simulation skipped")
else:
    print("fewer than two attack families -- zero-day simulation skipped")

In [ ]:
# ---------------------------------------------------------------------
# 5.5  Deployment readiness: latency, size, persistence, scoring API
# ---------------------------------------------------------------------
import joblib

batch = X_test.iloc[:min(50_000, len(X_test))]
t0 = time.perf_counter(); FINAL_MODEL.predict_proba(batch); batch_s = time.perf_counter() - t0

single = X_test.iloc[[0]]
FINAL_MODEL.predict_proba(single)                                   # warm-up
t0 = time.perf_counter()
for _ in range(100):
    FINAL_MODEL.predict_proba(single)
single_ms = (time.perf_counter() - t0) / 100 * 1000

artefact = Path(CFG.artefact_dir) / "xyz_threat_detector.joblib"
joblib.dump({
    "model": FINAL_MODEL,
    "features": FEATURES,
    "engineered_features": NEW_FEATURES,
    "threshold": FINAL_THRESHOLD,
    "label_semantics": {0: "BENIGN", 1: "ATTACK/FRAUD"},
    "trained_on": DATA_SOURCE,
    "sklearn_version": sklearn.__version__,
    "config": vars(CFG),
}, artefact, compress=3)

show("Deployment profile", pd.DataFrame([
    {"metric": "batch throughput", "value": f"{len(batch) / batch_s:,.0f} flows/s"},
    {"metric": "batch latency (50k flows)", "value": f"{batch_s:.2f} s"},
    {"metric": "single-flow latency", "value": f"{single_ms:.2f} ms"},
    {"metric": "artefact size", "value": f"{artefact.stat().st_size / 1024 ** 2:.1f} MiB"},
    {"metric": "features required at inference", "value": f"{len(FEATURES)}"},
    {"metric": "operating threshold", "value": f"{FINAL_THRESHOLD:.3f}"},
]).set_index("metric"))


def score_flow(flow: dict, model=FINAL_MODEL, threshold: float = FINAL_THRESHOLD) -> dict:
    """Score one raw flow record end-to-end (cleaning -> engineering -> probability -> verdict).

    Mirrors the production contract: raw CICIDS-style dict in, decision + explanation out.
    """
    frame = normalise_columns(pd.DataFrame([flow]))
    frame, _ = engineer_features(frame)
    for col in FEATURES:                                  # align schema; absent -> 0
        if col not in frame.columns:
            frame[col] = 0.0
    matrix = frame[FEATURES].astype("float32").replace([np.inf, -np.inf], 0).fillna(0)
    probability = float(model.predict_proba(matrix)[:, 1][0])
    drivers = (importance_table.head(5).index.tolist()
               if "importance_table" in globals() else FEATURES[:5])
    return {
        "attack_probability": round(probability, 6),
        "verdict": "ATTACK" if probability >= threshold else "BENIGN",
        "threshold": threshold,
        "confidence_band": ("high" if abs(probability - threshold) > 0.3 else "review"),
        "top_drivers": drivers,
    }


example = featured.iloc[0][[c for c in featured.columns if c not in
                            {"is_attack", "source_file"}]].to_dict()
print("\nsingle-flow scoring API demo")
print(f"true label : {example.get(LABEL_COL)}")
for key, value in score_flow(example).items():
    print(f"  {key:<20}: {value}")
print(f"\nartefact saved -> {artefact}")

### 5.6 Monitoring and retraining plan (post-deployment)

| Concern | Signal to watch | Trigger / action |
|---|---|---|
| **Data drift** | PSI / KS distance per feature vs the training reference, weekly | PSI > 0.2 on any top-10 feature → investigate; > 0.3 → retrain |
| **Concept drift** | Precision measured on analyst dispositions (their verdicts are the true labels) | precision drop > 5 points over 2 weeks → recalibrate threshold, then retrain |
| **Alert volume** | Alerts/hour vs SOC capacity | sustained breach of capacity → raise threshold (documented, reversible change) |
| **Silent failure** | Detection rate per attack family; purple-team replay of known attacks monthly | any family recall → 0 → treat as an incident |
| **Feedback loop bias** | Share of alerts actually reviewed; are ignored segments going unlabelled? | sample and label a random slice of *unalerted* traffic to keep the label set unbiased |
| **Model integrity** | Artefact hash + signature, input-range checks | mismatch → fail closed to the previous known-good model |

Cadence: scheduled retrain every 4 weeks on a rolling 6-month window, plus event-driven retrains.
Every release is shadow-deployed (scored in parallel, no blocking) for one week and compared against
the incumbent before promotion.

---
# 6. AI Ethics Considerations  *(5 marks)*

A threat-detection model is a system that *accuses*. Its errors are not abstract: a false positive
can lock a customer out of their account or get an employee investigated; a false negative can mean
a breach that harms thousands of people who never consented to anything. The analysis below is
organised around the harms, not around the algorithm.

## 6.1 Fairness and non-discrimination

**No protected attribute is used as a feature — but proxies exist.** The feature set is flow
telemetry, yet several fields correlate with people and groups:

| Feature | Hidden proxy risk | Mitigation applied |
|---|---|---|
| Source/destination IP (excluded here) | Identifies a person, a household, a country, an ISP | Never used as a feature; IPs are pseudonymised in storage |
| Destination Port | Correlates with assistive/alternative software, VPN use, or a specific department's tooling | Raw port dropped (§1.8); only broad service categories retained |
| Flow timing (`Idle`, `Active`, IAT) | Correlates with working hours → shift workers, carers, users in other time zones, users with disabilities who browse more slowly | Timing used only in shape-invariant/ratio form; no absolute time-of-day feature |
| Packet size / duration | Correlates with low-bandwidth regions, older devices, satellite links | Log/ratio transforms reduce absolute-bandwidth dependence |

**Concrete risk:** users on slow or high-latency connections produce flows that *look* like slow-rate
DoS. A model optimised only for aggregate PR-AUC can shift its errors onto that minority. Section 6.3
measures error rates per traffic segment and per attack family, because "fair overall" is not fair.

**Equality of coverage matters too.** In §5.1 the macro-F1 is deliberately reported next to the
weighted F1: if a rare attack family is never detected, the victims of that specific attack are
systematically less protected than everyone else, even though the headline metrics look excellent.

## 6.2 Other ethical dimensions

**Transparency & explainability.** Analysts and affected users are owed a reason, not a score.
§5.2 provides global explanations (permutation importance) and `score_flow()` returns the top drivers
for each decision. A blocked user must be able to ask *why*, and a human must be able to answer.

**Accountability & human oversight.** The model is decision *support*. Recommended policy: automated
blocking only for the highest-confidence band on unambiguous families (e.g. volumetric DoS);
everything else creates a ticket for human adjudication. Every automated action is logged with model
version, input hash, probability, threshold and outcome, so decisions are auditable and reversible,
and there is a named owner accountable for the system's behaviour.

**Privacy & data minimisation.** Flow metadata is still personal data under GDPR/PDPA — it reveals
who talked to whom, when and how much. Principles applied: collect metadata not payload; pseudonymise
IPs; define a retention limit (e.g. 90 days raw, aggregates thereafter) and enforce deletion;
restrict access by role; run a DPIA before deployment. Training data should not be shipped inside the
model artefact.

**Consent and workplace surveillance.** The same model that protects a corporate network can profile
employees. Ethical deployment requires notifying monitored users, limiting use to security purposes
only (no productivity scoring — purpose limitation), and prohibiting repurposing without a new
assessment.

**Adversarial robustness & security of the model.** Attackers adapt. The model is exposed to
evasion (padding packets, pacing traffic to mimic benign IAT), poisoning (feeding mislabelled
"benign" attacks into the retraining loop), and model extraction via probing. Mitigations: keep the
anomaly detector of §5.4 as a second, differently-failing layer; never auto-retrain on unreviewed
labels; rate-limit and monitor scoring APIs; run adversarial replay tests before each release.

**Dataset limitations and label bias.** CICIDS-style data is generated in a controlled lab: attack
traffic is scripted, benign traffic is synthetic, and the attack mix reflects the researchers' choices
rather than the threats XYZ's customers face. Labels come from the capture plan, so mislabelled or
mistimed flows are systematic, not random. Consequences: reported metrics are an **upper bound**;
performance on live traffic will be lower; and the model may encode "attack = the tools used in 2017".
Honest reporting of this limitation is itself an ethical obligation.

**Dual use.** A model that identifies attack traffic also identifies censorship-circumvention or
whistleblower traffic as "anomalous". XYZ should place contractual limits on use for political
targeting, and refuse deployments whose purpose is suppressing lawful speech.

**Regulatory framing.** Under the EU AI Act, cybersecurity monitoring of employees can fall into
high-risk territory (workplace monitoring), pulling in obligations on risk management,
data governance, logging, human oversight and technical documentation — the model card in §6.4 is the
first step towards that documentation.

**Environmental cost.** Hyperparameter search is compute-hungry. We used random search with a fixed
budget on a stratified subsample rather than an exhaustive grid (§4), which is both a scientific and
an environmental choice.

In [ ]:
# ---------------------------------------------------------------------
# 6.3  Fairness audit: is the error burden shared evenly across segments?
# ---------------------------------------------------------------------
# Protected attributes are absent by design, so we audit the operationally meaningful proxies:
# service category (which application/user group), traffic volume (bandwidth-poor users) and
# attack family (are victims of rare attacks protected as well as victims of common ones?).
audit_frame = pd.DataFrame({"y_true": y_test.to_numpy(), "y_pred": FINAL_PRED,
                            "proba": FINAL_PROBA, "family": ymulti_test.to_numpy()})

service_flags = [c for c in ["fe_port_is_web", "fe_port_is_encrypted", "fe_port_is_remote_admin",
                             "fe_port_is_file_transfer", "fe_port_is_infra"] if c in X_test.columns]
service = pd.Series("other", index=range(len(X_test)))
for flag in service_flags:
    service[X_test[flag].to_numpy() == 1] = flag.replace("fe_port_is_", "")
audit_frame["service"] = service.to_numpy()

vol_col = "fe_total_bytes" if "fe_total_bytes" in X_test.columns else FEATURES[0]
audit_frame["volume_band"] = quartile_bands(
    X_test[vol_col].to_numpy(), ["lowest 25%", "low-mid", "mid-high", "highest 25%"]).astype(str)


def segment_report(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    """Per-segment recall (protection level) and FPR (burden of false accusation)."""
    rows = []
    for seg, part in frame.groupby(by, observed=True):
        tn, fp, fn, tp = confusion_matrix(part["y_true"], part["y_pred"], labels=[0, 1]).ravel()
        rows.append({
            by: str(seg), "n": len(part),
            "attack_rate": part["y_true"].mean(),
            "recall": tp / max(tp + fn, 1),
            "fpr": fp / max(fp + tn, 1),
            "precision": tp / max(tp + fp, 1),
            "alert_rate": part["y_pred"].mean(),
        })
    return pd.DataFrame(rows).set_index(by).sort_values("fpr", ascending=False)


for dimension in ["service", "volume_band"]:
    report = segment_report(audit_frame, dimension)
    show(f"Fairness audit by {dimension}", report.round(4))
    spread = report["fpr"].max() - report["fpr"].min()
    ratio = report["fpr"].max() / max(report["fpr"].min(), 1e-9)
    print(f"-> FPR spread across {dimension}: {spread:.4f} "
          f"(worst/best ratio {ratio:,.1f}x). A ratio far above ~1.5 means one group carries a "
          f"disproportionate share of false accusations and needs a segment-specific threshold "
          f"or targeted data collection.")

In [ ]:
# ---------------------------------------------------------------------
# 6.3b  Visualising the disparity + protection gap across attack families
# ---------------------------------------------------------------------
svc = segment_report(audit_frame, "service")
fam = (segment_report(audit_frame[audit_frame["y_true"] == 1], "family")
       .sort_values("recall"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
xs = np.arange(len(svc))
axes[0].bar(xs - 0.2, svc["fpr"], 0.4, label="false-positive rate", color="#e63946")
axes[0].bar(xs + 0.2, svc["recall"], 0.4, label="recall", color="#2a9d8f")
axes[0].set_xticks(xs); axes[0].set_xticklabels(svc.index, rotation=25, ha="right")
axes[0].set_title("Error burden by service category")
axes[0].legend(fontsize=8)

axes[1].barh(fam.index, fam["recall"], color="#457b9d")
axes[1].axvline(audit_frame.loc[audit_frame["y_true"] == 1, "y_pred"].mean(), ls="--", c="k",
                lw=1, label="overall recall")
axes[1].set_xlabel("recall (protection level)")
axes[1].set_title("Protection gap: recall per attack family")
axes[1].tick_params(labelsize=7)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

worst = fam.index[0]
print(f"Least-protected attack family: {worst!r} (recall {fam['recall'].iloc[0]:.2%}, "
      f"n={int(fam['n'].iloc[0])}).")
print("""
Remediation options, in order of preference:
1. collect / label more examples of the under-detected family (fix the data, not the metric);
2. add domain features that specifically characterise it;
3. train a specialist detector for it and OR the alerts together;
4. only then consider segment-specific thresholds -- and document them, since different thresholds
   for different groups is itself an ethical decision that must be justified and reviewed.
""")

In [ ]:
# ---------------------------------------------------------------------
# 6.4  Model card -- the accountability artefact that ships with the model
# ---------------------------------------------------------------------
final_metrics = evaluate(y_test, FINAL_PROBA, FINAL_THRESHOLD, FINAL_NAME)

MODEL_CARD = f"""
================================================================================
MODEL CARD -- XYZ Cybersecurity Network Threat / Fraud Detector
================================================================================
1. MODEL DETAILS
   Architecture      : {TUNE_ARCH} ({FINAL_NAME}), scikit-learn {sklearn.__version__}
   Task              : binary classification of network flows (BENIGN vs ATTACK/FRAUD)
                       + multi-class attack attribution (Section 5.1)
   Version / date    : 1.0 (notebook build)
   Owner             : XYZ Cybersecurity, Detection Engineering (named owner accountable)
   Inputs            : {len(FEATURES)} numeric flow features ({len(NEW_FEATURES)} engineered)
   Output            : P(attack) in [0,1]; alert if p >= {FINAL_THRESHOLD:.3f}

2. INTENDED USE
   In scope          : triage support inside a monitored SOC; analyst-facing alerting;
                       automated blocking ONLY for the high-confidence band on volumetric attacks.
   Out of scope      : employee productivity scoring; law-enforcement evidence; any use where a
                       single model output causes irreversible action without human review;
                       deployment on traffic types absent from the training distribution.

3. TRAINING DATA
   Source            : {DATA_SOURCE} CICIDS-style flow exports ({', '.join(CFG.dataset_files)})
   Rows / features   : {len(featured):,} flows (post-cleaning) / {len(FEATURES)} features
   Class balance     : {y.mean():.2%} attack, {1 - y.mean():.2%} benign
   Known limitations : lab-generated traffic; scripted attacks; synthetic benign behaviour;
                       attack mix reflects the capture plan, not XYZ's live threat landscape.

4. PERFORMANCE (hold-out test set, threshold {FINAL_THRESHOLD:.3f})
   Recall            : {final_metrics['recall']:.4f}      (attacks caught)
   Precision         : {final_metrics['precision']:.4f}      (alerts that are real)
   F1 / MCC          : {final_metrics['f1']:.4f} / {final_metrics['mcc']:.4f}
   PR-AUC / ROC-AUC  : {final_metrics['pr_auc']:.4f} / {final_metrics['roc_auc']:.4f}
   FPR / FNR         : {final_metrics['fpr']:.4f} / {final_metrics['fnr']:.4f}
   Confusion matrix  : TP={final_metrics['tp']:,} FP={final_metrics['fp']:,} """ + \
f"""FN={final_metrics['fn']:,} TN={final_metrics['tn']:,}
   Disaggregated     : see Section 6.3 (per service category, volume band, attack family)

5. ETHICAL CONSIDERATIONS
   Protected attributes : none used; proxy risks documented and audited (Section 6.1/6.3)
   Privacy              : metadata only, IP pseudonymisation, defined retention, DPIA required
   Human oversight      : analyst adjudication for all but the high-confidence band; full audit log
   Explainability       : global importances + per-decision drivers via score_flow()
   Known failure modes  : low-bandwidth/high-latency users resemble slow-rate DoS; rare attack
                          families under-detected; adversarial evasion by traffic shaping;
                          degradation under distribution shift (see Section 5.3)

6. MAINTENANCE
   Monitoring        : drift (PSI), analyst-verified precision, alert volume, per-family recall
   Retraining        : every 4 weeks on a rolling window + event-driven; shadow-deploy before promotion
   Rollback          : previous artefact retained; fail closed to last known-good model
================================================================================
"""
print(MODEL_CARD)

card_path = Path(CFG.artefact_dir) / "MODEL_CARD.md"
card_path.write_text(MODEL_CARD)
print(f"model card written -> {card_path}")

---
# 7. Conclusion, Limitations and Next Steps

*(Documentation & code quality, 3 marks: every section states its intent before the code, every
helper carries a docstring explaining **why** not just what, all experiment knobs live in the single
frozen `CFG` object, seeds are fixed for reproducibility, preprocessing is wrapped in `Pipeline`s to
prevent leakage, and each result is printed or plotted rather than asserted.)*

In [ ]:
# ---------------------------------------------------------------------
# 7.1  Consolidated results
# ---------------------------------------------------------------------
summary = pd.DataFrame([
    evaluate(y_test, np.zeros(len(y_test)), 0.5, "always-BENIGN (trap)"),
    evaluate(y_test, proba_test, 0.5, f"{BEST_NAME} @0.50 (default)"),
    evaluate(y_test, proba_test, OPERATING_THRESHOLD, f"{BEST_NAME} @cost-optimal"),
    evaluate(y_test, FINAL_PROBA, FINAL_THRESHOLD, f"FINAL: {FINAL_NAME} {TUNE_ARCH}"),
]).set_index("model")

show("End-to-end results (hold-out test set)",
     summary[["threshold", "accuracy", "balanced_acc", "precision", "recall", "f1", "mcc",
              "pr_auc", "roc_auc", "fpr", "tp", "fp", "fn", "expected_cost"]].T)

print(f"\nData source used for this run : {DATA_SOURCE.upper()}"
      f"{'  (results are structural only -- re-run with the real CSVs)' if DATA_SOURCE != 'real' else ''}")
print(f"Flows analysed                : {len(featured):,}")
print(f"Features (engineered)         : {len(FEATURES)} ({len(NEW_FEATURES)})")
print(f"Selected architecture         : {BEST_NAME} -> tuned as {TUNE_ARCH}")
print(f"Operating threshold           : {FINAL_THRESHOLD:.3f} "
      f"(cost ratio FN:FP = {CFG.cost_false_negative:.0f}:{CFG.cost_false_positive:.0f})")
print(f"Artefacts                     : {sorted(p.name for p in Path(CFG.artefact_dir).iterdir())}")

## What was done, and what it means

1. **Data understanding & preparation (§1)** — three flow exports merged into one frame; the audit
   surfaced duplicated rows, ±∞ in the rate columns, constant columns and a duplicated header field,
   all fixed in a logged, deterministic cleaning pipeline. Feature engineering added domain-driven
   ratio, rate, asymmetry, burstiness, flag-density and service-category features; §5.2 confirms
   several of them rank among the most important predictors, so this was not decoration.
2. **Model selection (§2)** — a diverse shortlist screened by stratified CV on PR-AUC, with a dummy
   floor and an interpretable linear baseline to quantify what non-linearity actually buys. The
   imbalance strategy was chosen by experiment, not by habit.
3. **Performance measurement (§3)** — recall, precision, F1, MCC, balanced accuracy, PR-AUC, ROC-AUC,
   FPR and an explicit cost function, with the always-BENIGN trap shown side by side. The operating
   threshold was derived from business cost on training data, never from the test set.
4. **Hyperparameter tuning (§4)** — randomised search over 6–8 interacting hyperparameters scored on
   PR-AUC, with a train-vs-CV gap plot to expose overfitting candidates and a marginal-sensitivity
   table showing which knobs actually mattered. Tuned vs untuned is compared like-for-like.
5. **Extra features (§5)** — attack attribution, permutation/SHAP explainability, a capture-based
   generalisation stress test, unsupervised zero-day coverage, and a deployment profile with a
   single-flow scoring API plus a monitoring and retraining plan.
6. **AI ethics (§6)** — proxy-attribute analysis, a quantitative disparity audit across service
   categories, bandwidth bands and attack families, plus privacy, consent, adversarial-robustness,
   dual-use, dataset-bias and regulatory considerations, all consolidated into a model card.

## Honest limitations

* **Benchmark, not reality.** Lab-generated traffic with scripted attacks inflates every metric here;
  live performance will be materially lower. §5.3 is the most realistic number in the notebook.
* **No temporal validation.** Flow records carry no reliable timestamp in this export, so a true
  time-based split (train on the past, test on the future) is approximated by capture hold-out.
* **Label trust.** Labels derive from the capture plan; systematic mislabelling cannot be detected
  from the features alone.
* **Concept drift is guaranteed.** Attacker tooling changes faster than retraining cycles; static
  metrics decay. The monitoring plan in §5.6 is not optional.
* **Probability calibration** was not formally assessed; if downstream systems consume the
  probability as a risk score, add `CalibratedClassifierCV` and a reliability diagram.

## Next steps

1. Validate on XYZ's own captured traffic and re-measure the fairness audit on real user segments.
2. Add calibration + a reliability diagram so thresholds carry a probabilistic meaning.
3. Stream the feature engineering into the collector so training and serving share one code path
   (a feature store prevents train/serve skew).
4. Adversarial testing: replay evasion (packet padding, IAT pacing) and quantify the recall drop.
5. Human-factors evaluation: measure analyst time-to-disposition with and without the model, since
   the real objective is faster correct decisions, not a higher F1.

## References

* Sharafaldin, I., Habibi Lashkari, A., & Ghorbani, A. (2018). *Toward Generating a New Intrusion
  Detection Dataset and Intrusion Traffic Characterization.* ICISSP. (CICIDS-2017 dataset & features)
* Bergstra, J., & Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html). *JMLR* 13.
* Saito, T., & Rehmsmeier, M. (2015). *The Precision-Recall Plot Is More Informative than the ROC Plot
  on Imbalanced Datasets.* PLOS ONE.
* Mitchell, M. et al. (2019). [Model Cards for Model Reporting](https://arxiv.org/abs/1810.03993). *FAT\**.
* Chawla, N. et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique.* JAIR 16.
* Breiman, L. (2001). *Random Forests.* *Machine Learning* 45(1).
* Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.* *JMLR* 12.